# FinTrace 360: AMLSim Exploratory Data Analysis (EDA)
**Module:** Financial Analytics & Visual Intelligence  
**Objective:** Analyze transaction distributions, examine daily volume dynamics, compare banking entities, and evaluate network topology signatures of SAR alerts.


In [ ]:
import sys
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

BASE_DIR = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(BASE_DIR))

sns.set_theme(style="whitegrid", palette="muted")
plt.rcParams.update({"font.sans-serif": "Arial", "figure.autolayout": True, "figure.dpi": 150})

df_acc = pd.read_csv(BASE_DIR / "data/processed/amlsim_accounts_clean.csv")
df_tx = pd.read_csv(BASE_DIR / "data/processed/amlsim_transactions_clean.csv")
df_feats = pd.read_csv(BASE_DIR / "data/processed/amlsim_account_features.csv")


## 1. Transaction Amount Distribution


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
sns.histplot(df_tx['amount'], bins=40, kde=True, ax=axes[0], color="#1f77b4")
axes[0].set_title("Transaction Amount Distribution")

sns.boxplot(x="transaction_type", y="amount", data=df_tx, ax=axes[1], palette="Blues_r")
axes[1].set_title("Transaction Amount by Payment Rail")
plt.show()


## 2. Daily Transaction Trajectory (149 Operating Days)


In [ ]:
daily_summary = df_tx.groupby("step_day").agg(
    total_tx=('transaction_id', 'count'),
    gross_volume=('amount', 'sum'),
    alerted_tx=('is_alerted_tx', 'sum')
).reset_index()

plt.figure(figsize=(14, 5))
plt.plot(daily_summary['step_day'], daily_summary['total_tx'], label='Total Transactions', color='#2b5c8f')
plt.plot(daily_summary['step_day'], daily_summary['alerted_tx'], label='Alerted Transactions', color='#d62728', linewidth=2)
plt.title("Daily Transaction Count & Alert Trajectory")
plt.xlabel("Operating Step (Day)")
plt.ylabel("Transaction Count")
plt.legend()
plt.show()


## 3. Network Degree Analysis (Fan-In vs Fan-Out)


In [ ]:
sample_feats = df_feats.sample(n=min(5000, len(df_feats)), random_state=42)
plt.figure(figsize=(10, 6))
sns.scatterplot(
    data=sample_feats,
    x="out_tx_count",
    y="in_tx_count",
    hue="is_sar_alert",
    palette={0: "#4c72b0", 1: "#d62728"},
    alpha=0.6,
    s=35
)
plt.title("Account Network Inflow vs. Outflow Count (SAR Hub Detection)")
plt.xlabel("Outgoing Transactions (Fan-Out)")
plt.ylabel("Incoming Transactions (Fan-In)")
plt.show()


## 4. Key Financial Risk Correlation Matrix


In [ ]:
corr_cols = [
    "initial_balance", "total_tx_count", "total_volume", "net_flow",
    "unique_counterparties", "tx_velocity_per_day", "fan_in_index", "fan_out_index",
    "turnover_ratio", "is_sar_alert"
]
plt.figure(figsize=(10, 8))
sns.heatmap(df_feats[corr_cols].corr(), annot=True, fmt=".2f", cmap="coolwarm", square=True)
plt.title("Feature Correlation Matrix")
plt.show()
